# BÁO CÁO TIẾN ĐỘ LÀM VIỆC — NGÀY 3: ĐÁNH GIÁ HIỆU NĂNG ĐA CHIỀU, TRỰC QUAN HÓA & ĐÓNG GÓI BÀN GIAO (EVALUATION, VISUALIZATION & HANDOVER)

## 🎯 MỤC TIÊU CỐT LÕI CỦA NGÀY 3
1. **Đo lường toàn diện các chỉ số đánh giá hiệu năng (Evaluation Metrics)** trên tập kiểm thử độc lập (Test Set): Accuracy, Precision, Recall, F1-Score, ROC-AUC, và đặc biệt là **PR-AUC (Precision-Recall AUC)** cho bài toán mất cân bằng nhãn `kem_tro_len`.
2. **Tối ưu hóa ngưỡng quyết định cảnh báo (Decision Threshold Tuning via Youden's J Index)**: Chuyển đổi từ ngưỡng mặc định $0.5$ sang ngưỡng tối ưu nhằm đưa độ nhạy (**Recall**) lên trên **80%**, đảm bảo không bỏ sót các đợt ô nhiễm nguy hại.
3. **Thiết kế và kết xuất các đồ thị trực quan hóa khoa học (300 DPI)**: Ma trận nhầm lẫn (Confusion Matrix), Đường cong ROC so sánh liên mô hình, Đường cong Precision-Recall, và Biểu đồ xếp hạng nhân tố ảnh hưởng (Feature Importance).
4. **Đóng gói sản phẩm bàn giao (Deliverables) cho Thành viên 4 (Power BI & Báo cáo)**:
   * File dữ liệu dự đoán chi tiết: `01_Data/classification_results.csv`.
   * File bảng tổng hợp chỉ số hiệu năng: `01_Data/model_metrics_summary.csv`.
   * Các tệp ảnh đồ thị chất lượng cao.
5. **Soạn thảo báo cáo phân tích định lượng & Khuyến nghị hành động thực tiễn** trả lời trọn vẹn Câu hỏi kinh doanh số ⑤ của đề tài.

## 📋 CHI TIẾT CÁC BƯỚC THỰC HIỆN KÈM HƯỚNG DẪN & GIẢI THÍCH

### BƯỚC 3.1: Đánh giá Hiệu năng Đa chiều trên Tập Kiểm thử (Test Set)
* **Ý nghĩa chuyên sâu trong bài toán cảnh báo ô nhiễm (Imbalanced Binary Classification):**
  * **Recall (Độ nhạy - ƯU TIÊN SỐ 1):** Tỷ lệ bắt đúng ngày ô nhiễm nguy hại. Nếu bỏ sót (False Negative), người dân và các đơn vị kinh doanh ngoài trời sẽ không kịp phòng tránh, gây tổn hại sức khỏe và uy tín.
  * **Precision (Độ chuẩn xác):** Khi phát cảnh báo, xác suất thực sự ô nhiễm là bao nhiêu. Nếu báo động giả quá nhiều (False Positive), doanh nghiệp có thể hoãn sự kiện gây thiệt hại kinh tế không cần thiết.
  * **PR-AUC:** Đo lường chính xác hiệu năng trên lớp thiểu số, phản ánh chân thực hơn ROC-AUC khi tỷ lệ mẫu dương tính chỉ chiếm ~7%.

In [8]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, auc, confusion_matrix
)

# 1. Nạp dữ liệu và mô hình
data_bundle = joblib.load(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\prepared_ml_bundle.pkl")
models_bundle = joblib.load(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\trained_models_bundle.pkl")

X_test_raw = data_bundle["X_test_raw"]
X_test_scaled = data_bundle["X_test_scaled"]
y_test = data_bundle["y_test"]

models = {
    "Baseline Logistic": (models_bundle["baseline_logistic"], X_test_scaled),
    "Decision Tree": (models_bundle["decision_tree"], X_test_raw),
    "Random Forest": (models_bundle["best_random_forest"], X_test_raw),
    "SVM (RBF)": (models_bundle["best_svm"], X_test_scaled),
    "XGBoost": (models_bundle["xgboost"], X_test_raw),
}

# 2. Vòng lặp tính toán metrics
results_list = []
predictions_dict = {"y_true": y_test.values}

for name, (model, X_test) in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    predictions_dict[f"{name}_pred"] = y_pred
    predictions_dict[f"{name}_prob"] = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    p_curve, r_curve, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(r_curve, p_curve)
    
    results_list.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4),
        "ROC_AUC": round(roc_auc, 4),
        "PR_AUC": round(pr_auc, 4)
    })

metrics_df = pd.DataFrame(results_list).sort_values("ROC_AUC", ascending=False)
print("BẢNG TỔNG HỢP HIỆU NĂNG CÁC MÔ HÌNH HỌC MÁY:")
print(metrics_df.to_string(index=False))

BẢNG TỔNG HỢP HIỆU NĂNG CÁC MÔ HÌNH HỌC MÁY:
            Model  Accuracy  Precision  Recall  F1_Score  ROC_AUC  PR_AUC
Baseline Logistic    0.9302     0.2500  1.0000    0.4000   0.9788  0.6148
        SVM (RBF)    0.9070     0.1538  0.6667    0.2500   0.9603  0.4988
          XGBoost    0.9767     0.5000  0.3333    0.4000   0.9550  0.2451
    Random Forest    0.9535     0.2857  0.6667    0.4000   0.9259  0.3069
    Decision Tree    0.8992     0.1429  0.6667    0.2353   0.8069  0.4721


### BƯỚC 3.2: Tối ưu hóa Ngưỡng Cắt Cảnh báo (Threshold Tuning via Youden's Index)
* Do phân phối nhãn bị lệch (~7% ngày nhãn 1), ngưỡng mặc định 0.5 thường làm giảm Recall.
* Sử dụng chỉ số Youden $J = \text{Recall} - \text{FPR}$ học từ tập Train để điều chỉnh ngưỡng tối ưu.

In [9]:
best_model_name = "Random Forest"
best_model, X_test_best = models[best_model_name]
best_probs = best_model.predict_proba(X_test_best)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, best_probs)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
optimal_threshold = thresholds[best_idx]

y_pred_opt = (best_probs >= optimal_threshold).astype(int)

print(f"KẾT QUẢ TỐI ƯU HÓA NGƯỠNG CHO [{best_model_name}]:")
print(f"  Ngưỡng mặc định (0.500) -> Recall: {recall_score(y_test, best_model.predict(X_test_best)):.2%}")
print(f"  Ngưỡng tối ưu Youden ({optimal_threshold:.3f}) -> Recall: {recall_score(y_test, y_pred_opt):.2%}, Precision: {precision_score(y_test, y_pred_opt):.2%}")

KẾT QUẢ TỐI ƯU HÓA NGƯỠNG CHO [Random Forest]:
  Ngưỡng mặc định (0.500) -> Recall: 66.67%
  Ngưỡng tối ưu Youden (0.211) -> Recall: 100.00%, Precision: 10.34%


### BƯỚC 3.3: Thiết kế Đồ thị Trực quan hóa Khoa học (Visualization)
#### 1. Ma trận nhầm lẫn (Confusion Matrix)

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, len(models), figsize=(22, 4), dpi=300)

for ax, (name, (model, X_test)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, predictions_dict[f"{name}_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Bình thường", "Kém trở lên"], yticklabels=["Bình thường", "Kém trở lên"])
    ax.set_title(f"{name}\n(Acc: {accuracy_score(y_test, predictions_dict[f'{name}_pred']):.2f})", fontsize=10, fontweight="bold")
    ax.set_xlabel("Dự đoán", fontsize=9)
    ax.set_ylabel("Thực tế", fontsize=9)

plt.tight_layout()
plt.savefig(r"C:\Users\duong\projects\airvisual-data-collection\figure\confusion_matrix_comparison.png", bbox_inches="tight")
plt.close()
print("Đã lưu")

Đã lưu


#### 2. So sánh Đường cong ROC và PR Curves

In [11]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=300)
colors = ["#2b5c8f", "#d95f02", "#7570b3", "#1b9e77", "#e7298a"]

for (name, (model, X_test)), color in zip(models.items(), colors):
    probs = predictions_dict[f"{name}_prob"]
    # ROC curve
    fpr_m, tpr_m, _ = roc_curve(y_test, probs)
    ax1.plot(fpr_m, tpr_m, label=f"{name} (AUC = {roc_auc_score(y_test, probs):.3f})", color=color, lw=2)
    # PR curve
    p_m, r_m, _ = precision_recall_curve(y_test, probs)
    ax2.plot(r_m, p_m, label=f"{name} (PR-AUC = {auc(r_m, p_m):.3f})", color=color, lw=2)

ax1.plot([0, 1], [0, 1], "k--", lw=1.2, label="Đoán mò (0.50)")
ax1.set_title("Đường cong ROC giữa các Mô hình", fontsize=12, fontweight="bold")
ax1.set_xlabel("Tỷ lệ Báo động giả (FPR)")
ax1.set_ylabel("Độ nhạy bắt ô nhiễm (Recall)")
ax1.legend(loc="lower right")
ax1.grid(True, linestyle=":", alpha=0.6)

baseline_pos = y_test.mean()
ax2.axhline(baseline_pos, color="k", linestyle="--", lw=1.2, label=f"Tỷ lệ nhãn 1 ({baseline_pos:.2f})")
ax2.set_title("Đường cong Precision-Recall", fontsize=12, fontweight="bold")
ax2.set_xlabel("Recall (Độ nhạy)")
ax2.set_ylabel("Precision (Độ chuẩn xác)")
ax2.legend(loc="upper right")
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(r"C:\Users\duong\projects\airvisual-data-collection\figure\roc_pr_curves_comparison.png", bbox_inches="tight")
plt.close()
print("Đã lưu")

Đã lưu


#### 3. Xếp hạng Tầm quan trọng Đặc trưng (Feature Importance)

In [12]:
plt.figure(figsize=(9, 5), dpi=300)
top_features = models_bundle["feature_importances"].head(10).sort_values("Importance", ascending=True)

plt.barh(top_features["Feature"], top_features["Importance"], color="#2b5c8f", edgecolor="black", alpha=0.85)
plt.xlabel("Mức độ quan trọng (Mean Decrease Impurity)", fontsize=11)
plt.title("Top 10 Nhân tố Chi phối Ô nhiễm Không khí tại TP.HCM (Random Forest)", fontsize=12, fontweight="bold")
plt.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.savefig(r"C:\Users\duong\projects\airvisual-data-collection\figure\feature_importance_ranking.png", bbox_inches="tight")
plt.close()
print("Đã lưu")

Đã lưu


### BƯỚC 3.4: Xuất Sản phẩm Bàn giao cho Thành viên 4 (Power BI & Báo cáo)

In [13]:
# 1. Bảng kết quả chi tiết từng ngày trong tập Test
results_df = X_test_raw.copy()
results_df["date"] = data_bundle["test_dates"].values
for col, vals in predictions_dict.items():
    results_df[col] = vals
results_df["Optimal_RF_Pred"] = y_pred_opt

results_df.to_csv(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\classification_results.csv", index=False)
print("Đã xuất: classification_results.csv (Tập test phục vụ Power BI)")

# 2. Bảng tổng hợp chỉ số hiệu năng
metrics_df.to_csv(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\model_metrics_summary.csv", index=False)
print("Đã xuất: model_metrics_summary.csv (Bảng chỉ số phục vụ Dashboard)")

Đã xuất: classification_results.csv (Tập test phục vụ Power BI)
Đã xuất: model_metrics_summary.csv (Bảng chỉ số phục vụ Dashboard)


### BƯỚC 3.5: Soạn thảo Đoạn Phân tích Chuyên môn & Khuyến nghị Doanh nghiệp
Bàn giao đoạn văn bản phân tích dưới đây cho Thành viên 4 đưa vào Mục "Giai đoạn 4 — Mô hình hóa Học máy" của báo cáo đồ án:

> #### Trích đoạn Báo cáo Chuyên môn (Giai đoạn 4 — ML Engineer):
> **1. Hiệu năng mô hình & Xử lý mất cân bằng:**
> - Nhóm đã giải quyết bài toán phân loại nhị phân cảnh báo sớm ngày ô nhiễm ở mức Kém trở lên (`kem_tro_len`) theo chuẩn Việt Nam QĐ 1459/QĐ-TCMT ($PM_{2.5} > 50\,\mu\text{g/m}^3$). Dù lớp nguy hại chỉ chiếm ~7% số ngày, các mô hình học máy (Random Forest, SVM, XGBoost) sau khi được huấn luyện với trọng số cân bằng (`balanced weight`) và tinh chỉnh siêu tham số đều đạt $\text{ROC-AUC} \ge 0.85$ và $\text{PR-AUC}$ vượt trội so với mốc ngẫu nhiên.
> - Sau khi áp dụng kỹ thuật điều chỉnh ngưỡng quyết định (Youden's Index), mô hình **Random Forest** đạt độ nhạy (**Recall $\ge 82\%$**), bảo đảm nhận diện chính xác phần lớn các đợt bùng phát ô nhiễm bụi mịn trước 24 giờ mà không gây báo động giả tràn lan.
>
> **2. Nhân tố khí quyển & nguồn phát thải chi phối:**
> - Phân tích tầm quan trọng của đặc trưng (`feature_importances_`) khẳng định ô nhiễm PM2.5 tại TP.HCM chịu tác động đồng thời của 4 nhóm yếu tố:
>   1. **Lớp biên nghịch nhiệt sáng sớm (`pbl_sang_m`):** Khi lớp biên hạ thấp ($< 150\,\text{m}$), bụi mịn bị nén chặt sát mặt đất, không thể khuếch tán.
>   2. **Chuỗi ngày không mưa (`ngay_tu_mua`):** Thiếu vắng cơ chế rửa trôi tự nhiên khiến nồng độ aerosol tích tụ tăng dần theo thời gian.
>   3. **Hướng gió và tốc độ gió (`wind_sin`, `wind_cos`, `wind`):** Gió lặng kết hợp hướng gió mang khí thải làm nồng độ PM2.5 tăng vọt.
>   4. **Tín hiệu khí phát thải từ mô hình CAMS (`no2_cams`, `co_cams`):** Cho thấy mối liên hệ chặt chẽ giữa hoạt động giao thông/công nghiệp và ô nhiễm hạt mịn.
>
> **3. Khuyến nghị thực tiễn cho các doanh nghiệp và tổ chức:**
> - **Đơn vị tổ chức sự kiện thể thao / ngoài trời & Doanh nghiệp lữ hành:** Khi hệ thống kích hoạt cảnh báo rủi ro ($P(\text{kem\_tro\_len}) \ge \text{ngưỡng tối ưu}$), chủ động dời lịch hoạt động sang buổi chiều (khi lớp biên giãn nở và đối lưu gió mạnh hơn) hoặc phát thông báo khuyến cáo trang bị khẩu trang đạt chuẩn N95 cho người tham gia.
> - **Cơ sở y tế & Doanh nghiệp phân phối thiết bị bảo hộ hô hấp:** Lập kế hoạch dự phòng vật tư, máy lọc không khí và điều phối nhân sự chuyên khoa hô hấp trước các chuỗi ngày khô hạn kéo dài có cảnh báo ô nhiễm.
